# Shor's algorithm on real hardware: what factoring 15 does and does not prove

*Part of the QUEST Cryptography & Security series*

Shor's algorithm is the reason quantum computing has a place in security planning. It factors an $n$-bit integer in time polynomial in $n$, which breaks RSA, and the same machinery breaks Diffie-Hellman and elliptic-curve cryptography. Every few years a paper reports a number factored on a quantum device: 15, then 21, then larger ones. Read the methods sections and a pattern appears. Almost none of them ran the algorithm as written.

The usual shortcut is compilation. If you already know the period the algorithm is supposed to find, you can strip the circuit down to something a small device can run, and it will announce the right factors. Smolin, Smith and Vargo made the point sharply in 2013: with that shortcut, two coherent qubits suffice to "factor" any product of two distinct odd primes. The size of the number tells you nothing. What costs something is the length of the period.

This notebook builds order finding for $N = 15$ three ways. The first derives every gate from modular arithmetic and uses no knowledge of the period. The second tunes the same circuit to this particular instance. The third is a compiled circuit of the kind the demonstrations use, and we point it at a 2048-bit RSA modulus to show what it proves. We run the honest circuit and the compiled circuit on real processors, then apply the published surface-code resource estimates to RSA-2048.

**Learning objectives.** By the end of this notebook you will:

1. Reduce factoring to order finding and implement the classical half of the reduction.
2. Build controlled modular multiplication as a permutation derived from the arithmetic, and verify it against the classical map instead of trusting it.
3. Recover the period from a phase estimate by continued fractions, and explain why half the measurement outcomes are useless.
4. Demonstrate that a compiled circuit which already knows the period "factors" a 2048-bit modulus with four qubits.
5. Measure how far the honest circuit and the compiled circuit land from their ideal output on current hardware.
6. Apply the Gidney-Ekera resource estimates to RSA-2048 and state what they assume.

**What to bring in.** Quantum phase estimation, at the level of the QPE notebook in the Foundations & Algorithms series; this notebook reuses its inverse QFT and its reading of the phase register. From number theory: greatest common divisors, multiplicative order, and the Chinese remainder theorem. No cryptography background is assumed.

**Credit budget.** All simulator work is free. The hardware section runs 2 circuits on 3 backends at 1,000 shots each, for 6,000 shots total. At typical rates this is a few dollars in credits.

## From factoring to order finding

Shor's algorithm does not factor directly. It solves a different problem, and a classical reduction turns that solution into factors.

Pick $a$ coprime to $N$. The **order** of $a$ modulo $N$ is the smallest $r > 0$ with

$$a^r \equiv 1 \pmod N$$

Suppose we have $r$ and it happens to be even. Then $a^{r/2}$ is a square root of 1 modulo $N$, so

$$\left(a^{r/2} - 1\right)\left(a^{r/2} + 1\right) = a^r - 1 \equiv 0 \pmod N$$

meaning $N$ divides the product on the left. If additionally $a^{r/2} \not\equiv -1 \pmod N$, neither factor is divisible by $N$ on its own, so each must share part of $N$. Then $\gcd(a^{r/2} - 1, N)$ is a non-trivial factor, and a gcd is cheap. Two conditions can fail: $r$ can be odd, and $a^{r/2}$ can be $-1$. For $N$ with at least two distinct odd prime factors, a random $a$ fails both tests with probability at most $1/2$, so a handful of attempts suffices.

Everything above is classical. The only step a classical computer cannot do quickly is finding $r$.

The quantum part finds $r$ as a period. Define the unitary

$$U_a \lvert y \rangle = \lvert a y \bmod N \rangle$$

Its eigenvalues are $e^{2\pi i s / r}$ for $s = 0, 1, \dots, r-1$, with eigenvectors that are superpositions over the orbit of $a$. We cannot prepare an eigenvector without knowing $r$, but we do not need to: $\lvert 1 \rangle$ is an equal superposition of all $r$ of them. Phase estimation on $\lvert 1 \rangle$ therefore returns an estimate of $s/r$ for an $s$ drawn uniformly, and continued fractions recover $r$ from that estimate.

So the quantum circuit is phase estimation with $U_a$ in the controlled slot. The work is entirely in building controlled $U_a^{2^j}$, which is modular multiplication, which is arithmetic. That is where the cost lives, and it is where the shortcuts get taken.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import random
from math import gcd, log2
from fractions import Fraction

# Qiskit for circuit construction and local simulation
from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import QFTGate
from qiskit.quantum_info import Operator
from qiskit_aer import AerSimulator

# qBraid for unified device access
from qbraid.runtime import QbraidProvider

# Housekeeping
plt.rcParams['figure.dpi'] = 110
plt.rcParams['savefig.dpi'] = 110

sim = AerSimulator(seed_simulator=42)
BASIS = ['cx', 'rz', 'sx', 'x']       # a generic superconducting basis, for gate counting

N = 15                                 # the number to factor
A = 7                                  # the base whose order we will find
n = N.bit_length()                     # work register width, 4 qubits

print(f"Setup complete. Factoring N = {N} with base a = {A}, work register {n} qubits.")

## The classical half, done properly

Before any quantum circuit runs, a real implementation disposes of the easy cases. $N$ must not be even, not a perfect power, and not already divisible by a small prime. Then it draws $a$ at random and checks $\gcd(a, N)$, because a lucky draw factors $N$ outright and no quantum computer is needed.

We run those checks here so that the base we hand to the circuit is one the reduction genuinely requires a period for.

In [ ]:
def classical_precheck(N):
    """The tests a real implementation runs before touching a quantum device."""
    if N % 2 == 0:
        return f"even, factor 2"
    for b in range(2, int(log2(N)) + 1):
        root = round(N ** (1 / b))
        for cand in (root - 1, root, root + 1):
            if cand > 1 and cand ** b == N:
                return f"perfect power: {cand}^{b}"
    return None


print(f"N = {N}: precheck says {classical_precheck(N) or 'no easy factorisation, proceed'}")
print(f"gcd(a, N) = {gcd(A, N)}  (must be 1, else a already factors N)")

# The order, computed classically. We use this only to check the quantum answer,
# never to build the circuit.
r_true = next(r for r in range(1, N) if pow(A, r, N) == 1)
print(f"\nclassical order of {A} mod {N}: r = {r_true}")
print(f"a^(r/2) mod N = {pow(A, r_true // 2, N)}  (must not be N-1 = {N - 1})")
print(f"factors from the reduction: {gcd(pow(A, r_true // 2, N) - 1, N)} "
      f"and {gcd(pow(A, r_true // 2, N) + 1, N)}")

Two things to hold onto. `r_true` exists in this notebook only as a check on the quantum result; nothing in the circuit construction below reads it. And the reduction's arithmetic is trivial once $r$ is known, which is why the entire security question reduces to whether a device can find $r$.

## Modular multiplication is a permutation

$U_a \lvert y \rangle = \lvert ay \bmod N \rangle$ maps basis states to basis states. Since $\gcd(a, N) = 1$, multiplication by $a$ is a bijection on $\{0, 1, \dots, N-1\}$, and we leave the states $N \le y < 2^n$ fixed. So $U_a$ is a permutation matrix, and building it is a question of reversible logic rather than of general unitary synthesis.

We compute the permutation straight from the arithmetic, factor it into disjoint cycles, and factor each cycle into transpositions. No step consults the order.

In [ ]:
def perm_of(m, N, n):
    """The permutation y -> (m*y) mod N on n qubits, with y >= N left fixed."""
    p = list(range(2 ** n))
    for y in range(N):
        p[y] = (m * y) % N
    return p


def cycles_of(p):
    """Disjoint non-trivial cycles of a permutation given as a list."""
    seen, out = set(), []
    for start in range(len(p)):
        if start in seen or p[start] == start:
            continue
        c, x = [], start
        while x not in seen:
            seen.add(x)
            c.append(x)
            x = p[x]
        out.append(c)
    return out


def transpositions_of(p):
    """
    Factor into transpositions, in the order a circuit must apply them.
    The cycle (c0 c1 ... ck) equals (c0 ck) o ... o (c0 c1) under right-to-left
    function composition, so the circuit applies (c0 c1) first.
    """
    out = []
    for c in cycles_of(p):
        for i in range(1, len(c)):
            out.append((c[0], c[i]))
    return out


p7 = perm_of(A, N, n)
print(f"y      -> {A}y mod {N}")
for y in range(N + 1):
    print(f"{y:>2}     -> {p7[y]:>2}")
print(f"\ncycles: {cycles_of(p7)}")
print(f"transpositions: {transpositions_of(p7)}")

Three four-cycles and four fixed points. The fixed points are $0$, $5$, $10$ and the unused state $15$; the number $5$ is fixed because $7 \cdot 5 = 35 \equiv 5$, a consequence of $5$ sharing a factor with $15$. Nothing here required knowing that the order is 4, though in hindsight the cycle length is exactly that. Reading $r$ off a printed permutation is only possible because $N$ is small enough to print.

## Turning a transposition into gates

A transposition $\lvert x \rangle \leftrightarrow \lvert y \rangle$ becomes gates in three steps.

Let $d = x \oplus y$ and let $k$ be the index of any set bit of $d$. First, apply a CNOT from qubit $k$ onto every other qubit where $d$ is set. After this, $x$ and $y$ agree everywhere except bit $k$: for each targeted qubit $j$, the bit becomes $x_j \oplus x_k$ for $x$ and $(1 - x_j) \oplus (1 - x_k) = x_j \oplus x_k$ for $y$. Second, apply a multi-controlled $X$ on qubit $k$, with the other $n-1$ qubits as controls set to match the shared pattern. That flips bit $k$ for exactly those two states and swaps them. Third, undo the CNOTs.

The controlled version costs one extra control on the middle gate only. The outer CNOT layers are their own inverse, so when the control is off they cancel and the whole block is the identity.

The cost is one multi-controlled $X$ per transposition, and the number of transpositions grows with $2^n$. This construction is a lookup table, not an adder, and it does not scale. The polynomial constructions build modular multiplication out of modular adders instead; Beauregard's uses $2n + 3$ qubits and $O(n^3)$ gates. We use the lookup table because at $n = 4$ it is transparent and we can check it exactly, and we will compare its cost against the scalable ones later.

In [ ]:
def c_mult(m, N, n, label=None):
    """
    Controlled |y> -> |m*y mod N>.
    Qubit 0 is the control; qubits 1..n are the work register, qubit 1 the low bit.
    """
    qc = QuantumCircuit(1 + n, name=label or f"x{m} mod {N}")
    w = list(range(1, 1 + n))

    for x, y in transpositions_of(perm_of(m, N, n)):
        d = x ^ y
        k = (d & -d).bit_length() - 1          # lowest set bit of the difference
        fix = [j for j in range(n) if j != k and (d >> j) & 1]

        for j in fix:                           # collapse onto a single differing bit
            qc.cx(w[k], w[j])

        xp = x                                  # where x sits after that collapse
        for j in fix:
            xp ^= ((x >> k) & 1) << j

        ctrl_qubits = [j for j in range(n) if j != k]
        zeros = [w[j] for j in ctrl_qubits if not (xp >> j) & 1]

        for q in zeros:                         # controls fire on 0, so flip them
            qc.x(q)
        qc.mcx([0] + [w[j] for j in ctrl_qubits], w[k])
        for q in zeros:
            qc.x(q)

        for j in reversed(fix):                 # undo the collapse
            qc.cx(w[k], w[j])
    return qc


c_mult(A, N, n).draw('mpl', fold=110)

## Checking it against the arithmetic

The construction above is easy to get subtly wrong, and a wrong version can still produce the right factors. Reversing the transposition order builds multiplication by $a^{-1}$ rather than by $a$, which has the same order, so the algorithm still returns $r = 4$ and still factors 15. The only way to catch that is to compare the circuit's action against the classical map, state by state.

In [ ]:
def check_multiplier(m):
    """Compare the controlled circuit's action against (m*y) mod N on every basis state."""
    op = Operator(c_mult(m, N, n)).data
    p = perm_of(m, N, n)
    for y in range(2 ** n):
        # qubit 0 is the control, so the amplitude index is control + 2*y
        if abs(op[1 + 2 * p[y], 1 + 2 * y] - 1) > 1e-9:      # control on: y -> m*y
            return False, y
        if abs(op[2 * y, 2 * y] - 1) > 1e-9:                  # control off: identity
            return False, y
    return True, None


for m in (2, 4, 7, 8, 11, 13):
    ok, bad = check_multiplier(m)
    gates = c_mult(m, N, n).count_ops()
    print(f"multiply by {m:>2}: matches arithmetic = {ok}"
          f"{'' if ok else f' (fails at y={bad})'}   "
          f"mcx = {gates.get('mcx', 0)}, cx = {gates.get('cx', 0)}, x = {gates.get('x', 0)}")

All six bases coprime to 15 build correctly. This check is worth its runtime: it is the difference between an implementation and a circuit that happens to give the expected answer.

## The multipliers, and a warning about 15

Phase estimation needs controlled $U_a^{2^j}$, not $j$ copies of controlled $U_a$. The standard trick is to notice that $U_a^{2^j} = U_{a^{2^j} \bmod N}$ and to compute the multiplier classically by repeated squaring, which costs nothing. This is a genuine efficiency of the algorithm, present in every serious implementation, and it is not a shortcut of the kind we are criticising.

For $N = 15$ it has an awkward consequence.

In [ ]:
t = 2 * n                              # counting register width, the textbook prescription
mults = [pow(A, 2 ** j, N) for j in range(t)]

print(f"counting register: t = {t} qubits (textbook prescription 2n for n = {n})")
print(f"multipliers a^(2^j) mod N: {mults}")
print(f"non-trivial (not equal to 1): {sum(m != 1 for m in mults)} of {t}")

Six of the eight controlled multiplications are multiplication by 1, which is the identity, so they cost nothing at all. That is not the algorithm being cheap; it is 15 being small. The number of non-trivial multipliers is $\lceil \log_2 r \rceil$, and $r = 4$ here. For a 2048-bit modulus with a period of comparable size, all $4096$ of them are non-trivial and each is a full modular multiplication.

Every published factoring demonstration inherits this. When the reported circuit is small, the first question to ask is how much of the smallness is the algorithm and how much is the instance.

## Assembling order finding

Phase estimation, with the work register initialised to $\lvert 1 \rangle$ and the controlled multipliers in place of a controlled unitary power. The inverse QFT on the counting register turns the accumulated phases into a readable integer.

In [ ]:
def order_finding_circuit(t, a=A):
    """Phase estimation on U_a with a t-qubit counting register."""
    qc = QuantumCircuit(t + n, t)
    qc.h(range(t))                                  # uniform superposition of exponents
    qc.x(t)                                         # work register = |1>

    for j in range(t):
        m = pow(a, 2 ** j, N)
        if m == 1:                                  # identity, nothing to append
            continue
        qc.compose(c_mult(m, N, n), qubits=[j] + list(range(t, t + n)), inplace=True)

    qc.append(QFTGate(t).inverse(), range(t))       # read the phase
    qc.measure(range(t), range(t))
    return qc


honest = order_finding_circuit(t)
honest_t = transpile(honest, basis_gates=BASIS, optimization_level=2, seed_transpiler=7)

print(f"honest circuit: {honest.num_qubits} qubits")
print(f"  transpiled to {BASIS}: depth {honest_t.depth()}, "
      f"cx {honest_t.count_ops().get('cx', 0)}")

In [ ]:
SHOTS_SIM = 4096
counts_ideal = sim.run(transpile(honest, sim), shots=SHOTS_SIM).result().get_counts()

phases = {int(b, 2): c / SHOTS_SIM for b, c in counts_ideal.items()}

fig, ax = plt.subplots(figsize=(9.5, 4.5))
ax.bar(list(phases), list(phases.values()), width=2.5, color='#a02580')
for s in range(r_true):
    ax.axvline(s * 2 ** t / r_true, color='k', linestyle='--', linewidth=1.5, alpha=0.6)
ax.set_xlabel(f'measured integer $k$ (counting register, $t$ = {t})')
ax.set_ylabel('probability')
ax.set_title('Ideal order-finding output: peaks at $k = s\\,2^t/r$')
ax.set_xlim(-4, 2 ** t)
ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print("outcomes with probability above 1%:")
for k in sorted(phases):
    if phases[k] > 0.01:
        print(f"  k = {k:>3}   phase = {k / 2 ** t:.4f}   p = {phases[k]:.3f}")

Four peaks, at $k/2^t = 0, 1/4, 1/2, 3/4$, each with probability about $1/4$. The dashed lines are $s \cdot 2^t / r$ for the classically computed $r$, and the peaks sit exactly on them. They are exact here because $r = 4$ divides $2^t$; for a period that does not divide the register size the peaks broaden and continued fractions do real work.

## From a phase to a factor

Each outcome $k$ is an estimate of $s/r$ for an unknown $s$. Continued fractions give the best rational approximation with denominator below $N$, and that denominator is the candidate period. Two of the four outcomes are useless: $k = 0$ carries no information, and $k = 2^{t-1}$ gives $s/r = 1/2$, whose denominator is 2, and $7^2 = 4 \ne 1 \bmod 15$. A single run therefore succeeds about half the time, which is why the algorithm is stated with repetition.

In [ ]:
def factors_from_outcome(k, t, a=A):
    """Continued-fraction post-processing. Returns (r, factors, reason-if-failed)."""
    r = Fraction(k, 2 ** t).limit_denominator(N).denominator
    if r == 1:
        return r, None, 'denominator 1, no information'
    if pow(a, r, N) != 1:
        return r, None, f'a^{r} != 1 mod N, candidate period wrong'
    if r % 2:
        return r, None, 'odd period, reduction does not apply'
    x = pow(a, r // 2, N)
    if x == N - 1:
        return r, None, 'a^(r/2) = -1 mod N, reduction gives trivial factors'
    f = sorted({gcd(x - 1, N), gcd(x + 1, N)} - {1, N})
    return r, (f or None), None if f else 'gcd gave only trivial factors'


rows = []
for k in sorted(phases, key=lambda k: -phases[k])[:4]:
    r, f, why = factors_from_outcome(k, t)
    rows.append({'k': k, 'phase k/2^t': f'{k / 2 ** t:.3f}', 'p': f'{phases[k]:.3f}',
                 'candidate r': r, 'factors': ' x '.join(map(str, f)) if f else '-',
                 'outcome': 'success' if f else why})

pd.DataFrame(rows).set_index('k')

In [ ]:
success = sum(phases[k] for k in phases if factors_from_outcome(k, t)[1])
print(f"probability a single ideal run yields factors: {success:.3f}")
print(f"expected runs needed: {1 / success:.1f}")
print(f"probability of success within 5 runs: {1 - (1 - success) ** 5:.4f}")

## What a compiled demonstration does instead

Here is the shortcut. If you already know that $r = 4$, the work register no longer needs to hold residues modulo 15. It only needs to track position within the orbit $1 \to 7 \to 4 \to 13 \to 1$, which is four states, so two qubits. And $U_a$ restricted to that orbit is a cyclic shift: increment modulo 4. Increment modulo 4 is two gates.

The resulting circuit has the same output distribution as the honest one, so the post-processing works unchanged and the demonstration announces the factors 3 and 5. It is also a circuit that could not have been written without the answer. We use a three-qubit counting register so that the ideal distribution is peaked rather than uniform, which matters when we compare against hardware.

In [ ]:
def compiled_circuit(t_c=3, r_known=None):
    """
    Phase estimation where U is an increment mod r on a compressed orbit register.
    Requires r in advance; that is the entire point.
    """
    r_known = r_known or r_true
    w = int(np.ceil(log2(r_known)))                    # 2 qubits for r = 4
    qc = QuantumCircuit(t_c + w, t_c)
    qc.h(range(t_c))

    for j in range(t_c):
        for _ in range(2 ** j % r_known):              # controlled increment mod 4
            qc.ccx(j, t_c, t_c + 1)                    # high bit ^= low bit
            qc.cx(j, t_c)                              # low bit ^= 1

    qc.append(QFTGate(t_c).inverse(), range(t_c))
    qc.measure(range(t_c), range(t_c))
    return qc


T_C = 3
compiled = compiled_circuit(T_C)
compiled_t = transpile(compiled, basis_gates=BASIS, optimization_level=2, seed_transpiler=7)

print(f"compiled circuit: {compiled.num_qubits} qubits, "
      f"depth {compiled_t.depth()}, cx {compiled_t.count_ops().get('cx', 0)}")
print(f"honest circuit:   {honest.num_qubits} qubits, "
      f"depth {honest_t.depth()}, cx {honest_t.count_ops().get('cx', 0)}")

counts_c = sim.run(transpile(compiled, sim), shots=SHOTS_SIM).result().get_counts()
print(f"\ncompiled ideal outcomes: "
      f"{sorted((int(b, 2), round(c / SHOTS_SIM, 3)) for b, c in counts_c.items())}")
print(f"post-processing on the compiled output: "
      f"{factors_from_outcome(2, T_C)[1]}")

Two orders of magnitude in two-qubit gate count, for identical reported results. The honest circuit is the one that would still work if we did not know the answer, and it is the one no current device can run.

## The same four qubits, applied to RSA-2048

If the compiled circuit only encodes the period, then nothing ties it to $N = 15$. Any modulus with a base of order 4 gives the same circuit. So we build a 2048-bit RSA modulus, find a base of order 4 modulo it, and run the identical circuit.

Constructing that base requires the factorisation. That is the circularity, and it is worth seeing performed rather than described.

In [ ]:
def is_prime(x, rounds=32, rng=random):
    """Miller-Rabin."""
    if x < 2:
        return False
    for sp in (2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37):
        if x % sp == 0:
            return x == sp
    d, s = x - 1, 0
    while d % 2 == 0:
        d //= 2
        s += 1
    for _ in range(rounds):
        y = pow(rng.randrange(2, x - 1), d, x)
        if y in (1, x - 1):
            continue
        for _ in range(s - 1):
            y = y * y % x
            if y == x - 1:
                break
        else:
            return False
    return True


def prime_1_mod_4(bits, rng):
    """A prime p = 1 mod 4, so that a square root of -1 exists modulo p."""
    while True:
        p = rng.getrandbits(bits) | (1 << (bits - 1)) | 1
        p += (1 - p % 4) % 4
        if p % 4 == 1 and is_prime(p, rng=rng):
            return p


rng = random.Random(2024)
p, q = prime_1_mod_4(1024, rng), prime_1_mod_4(1024, rng)
N_rsa = p * q

# a = (a square root of -1) mod p, and 1 mod q. Then ord(a) = 4 modulo N_rsa.
while True:
    z = pow(rng.randrange(2, p - 1), (p - 1) // 4, p)
    if z * z % p == p - 1:
        break
a_rsa = (z * q * pow(q, -1, p) + p * pow(p, -1, q)) % N_rsa

print(f"N is {N_rsa.bit_length()} bits, the size of an RSA-2048 modulus")
print(f"order of a modulo N is 4: {pow(a_rsa, 4, N_rsa) == 1 and pow(a_rsa, 2, N_rsa) != 1}")

In [ ]:
# The identical 4,000-shot run of the identical compiled circuit, reused verbatim.
r_rsa = None
for b in sorted(counts_c, key=lambda b: -counts_c[b]):
    cand = Fraction(int(b, 2), 2 ** T_C).limit_denominator(N_rsa).denominator
    if cand % 2 == 0 and pow(a_rsa, cand, N_rsa) == 1:
        r_rsa = cand
        break

x = pow(a_rsa, r_rsa // 2, N_rsa)
f1, f2 = gcd(x - 1, N_rsa), gcd(x + 1, N_rsa)

print(f"period recovered from the quantum output: r = {r_rsa}")
print(f"gcd(a^(r/2) - 1, N) is a {f1.bit_length()}-bit factor")
print(f"gcd(a^(r/2) + 1, N) is a {f2.bit_length()}-bit factor")
print(f"the two factors multiply back to N: {f1 * f2 == N_rsa}")
print(f"both are prime: {is_prime(f1, rng=rng) and is_prime(f2, rng=rng)}")

A 2048-bit semiprime, split into its two 1024-bit prime factors, from the measurement record of a five-qubit circuit. RSA is not broken. Choosing $a$ used $p$ and $q$ directly, and the period was short because we arranged it to be. This is Smolin, Smith and Vargo's argument executed: the difficulty of a factoring demonstration is set by the length of the period found, not by the size of the number, and $\log_2 r = 2$ here.

The useful reading of a factoring claim is therefore: what was the period, how many controlled multiplications were non-trivial, and was the base chosen before or after the factorisation was known.

## On real processors

We submit both circuits to three devices. The honest circuit at $t = 8$ needs 12 qubits, which is more than some of these devices have, so for hardware we use $t = 4$. That is defensible for this instance and not in general: $t = 4$ suffices only because $r = 4$ divides 16 exactly, and choosing it uses a fact about the answer. It buys four qubits and almost no depth, since the cost sits in the two controlled multiplications rather than in the counting register. We report both so the trade is visible.

| Backend | Vendor | Modality | Why it matters here |
|---|---|---|---|
| `rigetti_ankaa_3` | Rigetti | Superconducting | Fast gates, lower two-qubit fidelity; the depth is the problem |
| `iqm_garnet` | IQM | Superconducting | Square lattice, so the multi-controlled gates need routing |
| `aqt_marmot` | AQT | Trapped ion | All-to-all coupling, high fidelity, slow gates |

*Device names are illustrative. Check `provider.get_devices()` for what is currently available in your account and substitute accordingly.*

**Note on queue times.** This is 6 hardware jobs. Depending on device load the submission cell may take minutes to hours.

In [ ]:
honest_hw = order_finding_circuit(4)                      # 8 qubits, fits every device
honest_hw_t = transpile(honest_hw, basis_gates=BASIS, optimization_level=2, seed_transpiler=7)

CIRCUITS = {
    'Honest (t=4)':  honest_hw,
    'Compiled (t=3)': compiled,
}

# Ideal support: the outcomes the circuit should produce, for scoring the hardware runs.
IDEAL_SUPPORT = {
    'Honest (t=4)':   {s * 2 ** 4 // r_true for s in range(r_true)},
    'Compiled (t=3)': {s * 2 ** T_C // r_true for s in range(r_true)},
}

for label, qc in CIRCUITS.items():
    tq = transpile(qc, basis_gates=BASIS, optimization_level=2, seed_transpiler=7)
    print(f"{label:16s} {qc.num_qubits} qubits, depth {tq.depth():>5}, "
          f"cx {tq.count_ops().get('cx', 0):>4}, ideal support {sorted(IDEAL_SUPPORT[label])}")
print(f"{'Honest (t=8)':16s} {honest.num_qubits} qubits, depth {honest_t.depth():>5}, "
      f"cx {honest_t.count_ops().get('cx', 0):>4}   (too wide for some devices)")

In [ ]:
provider = QbraidProvider()

# Instructor: update these IDs to match currently available hardware in your account.
BACKENDS = {
    'Rigetti Ankaa-3': 'rigetti_ankaa_3',
    'IQM Garnet':      'iqm_garnet',
    'AQT Marmot':      'aqt_marmot',
}

COLORS = {
    'Rigetti Ankaa-3': '#c63792',
    'IQM Garnet':      '#1a5285',
    'AQT Marmot':      '#2d7a4f',
}

SHOTS = 1000

devices = {name: provider.get_device(dev_id) for name, dev_id in BACKENDS.items()}
print("Configured backends:", list(devices))

In [ ]:
hw_counts = {name: {} for name in BACKENDS}

for name, device in devices.items():
    for label, qc in CIRCUITS.items():
        job = device.run(qc, shots=SHOTS)
        hw_counts[name][label] = job.result().data.get_counts()
        on_support = sum(c for b, c in hw_counts[name][label].items()
                         if int(b, 2) in IDEAL_SUPPORT[label]) / SHOTS
        print(f"{name:18s} {label:16s} shots on ideal support: {on_support:.3f}")

## The payoff

Two panels, same devices, same shot count. The dashed line is what a completely depolarised device would give: a uniform distribution over the counting register, which puts $r/2^t$ of its weight on the ideal support by accident.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12.5, 5), sharey=True)

for ax, (label, qc) in zip(axes, CIRCUITS.items()):
    t_lab = qc.num_clbits
    chance = len(IDEAL_SUPPORT[label]) / 2 ** t_lab

    names = list(BACKENDS)
    vals = [sum(c for b, c in hw_counts[nm][label].items()
                if int(b, 2) in IDEAL_SUPPORT[label]) / SHOTS for nm in names]

    ax.bar(range(len(names)), vals, color=[COLORS[nm] for nm in names], width=0.6)
    ax.axhline(1.0, color='k', linestyle='--', linewidth=2, alpha=0.6, label='Ideal')
    ax.axhline(chance, color='gray', linestyle=':', linewidth=2,
               label=f'Depolarised ({chance:.2f})')
    ax.set_xticks(range(len(names)))
    ax.set_xticklabels([nm.replace(' ', '\n') for nm in names], fontsize=9)
    ax.set_title(f'{label}\n{qc.num_qubits} qubits, '
                 f'{transpile(qc, basis_gates=BASIS, optimization_level=2, seed_transpiler=7).count_ops().get("cx", 0)} CX')
    ax.set_ylim(0, 1.12)
    ax.grid(alpha=0.3, axis='y')
    ax.legend(fontsize=9, loc='upper right')

axes[0].set_ylabel('fraction of shots on the ideal support')
fig.suptitle('Order finding for $N = 15$: honest circuit against compiled circuit', y=1.0)
plt.tight_layout()
plt.show()

The expected shape is a compiled circuit sitting near the ideal line on every device and an honest circuit sitting near the depolarised line. If the honest bars land at chance level, the device produced no information about the period at all, and any factors recovered from that run came from the post-processing being handed a lucky uniform sample rather than from the quantum computation.

That last point deserves emphasis, because it is the trap. Post-processing that tries several outcomes will eventually hit a usable $k$ even when the device output is pure noise, since $N = 15$ has only four candidate periods. A demonstration that reports "the factors 3 and 5 were recovered" without reporting the distribution has told you nothing about whether the circuit worked.

## Summary

In [ ]:
rows = []
for name in BACKENDS:
    for label, qc in CIRCUITS.items():
        t_lab = qc.num_clbits
        counts_hw = hw_counts[name][label]
        chance = len(IDEAL_SUPPORT[label]) / 2 ** t_lab
        on_support = sum(c for b, c in counts_hw.items()
                         if int(b, 2) in IDEAL_SUPPORT[label]) / SHOTS
        best = max(counts_hw, key=counts_hw.get)
        r_hw, f_hw, _ = factors_from_outcome(int(best, 2), t_lab)
        rows.append({
            'Backend': name,
            'Circuit': label,
            'CX': transpile(qc, basis_gates=BASIS, optimization_level=2,
                            seed_transpiler=7).count_ops().get('cx', 0),
            'On ideal support': f'{on_support:.3f}',
            'Chance level': f'{chance:.3f}',
            'Signal above chance': f'{(on_support - chance) / (1 - chance):+.3f}',
            'Most likely k': int(best, 2),
            'Factors from it': ' x '.join(map(str, f_hw)) if f_hw else '-',
        })

pd.DataFrame(rows).set_index(['Backend', 'Circuit'])

The "signal above chance" column rescales the result so that 1.0 is an ideal device and 0.0 is a device that returned nothing. It is the column to quote. The "factors from it" column is the one a press release would quote, and comparing the two across rows shows why that is a mistake.

## What RSA-2048 actually costs

Extrapolating from 15 is hopeless, so we use the published estimates instead. Gidney and Ekera (2019) give the resource counts for factoring an $n$-bit RSA modulus with a surface-code machine:

$$\text{logical qubits} = 3n + 0.002\, n \log_2 n \qquad \text{Toffoli gates} = 0.3\, n^3 + 0.0005\, n^3 \log_2 n$$

Their physical-layer figure for $n = 2048$ is **20 million noisy qubits running for 8 hours**, assuming a physical gate error rate of $10^{-3}$, a surface-code cycle time of 1 microsecond, a reaction time of 10 microseconds, and a planar grid with nearest-neighbour connectivity.

Gidney (2025) revisits the same problem with the same hardware assumptions and reaches **under 1 million noisy qubits in under a week**, a twentyfold reduction in qubits bought partly with a longer runtime and partly with a Toffoli count reduced by more than 100x over the intermediate construction of Chevignard, Fouque and Schrottenloher. Nothing about the hardware changed. The algorithms did.

In [ ]:
def logical_qubits(nbits):
    return 3 * nbits + 0.002 * nbits * log2(nbits)


def toffoli_count(nbits):
    return 0.3 * nbits ** 3 + 0.0005 * nbits ** 3 * log2(nbits)


sizes = [4, 15, 128, 512, 1024, 2048, 4096]
est = pd.DataFrame({
    'n (bits)': sizes,
    'logical qubits': [f'{logical_qubits(s):,.0f}' for s in sizes],
    'Toffoli gates': [f'{toffoli_count(s):.2e}' for s in sizes],
})
print(est.to_string(index=False))

print(f"\nFor N = 15 (n = {n}): {logical_qubits(n):.0f} logical qubits, "
      f"{toffoli_count(n):.0f} Toffolis.")
print(f"Our lookup-table circuit used {honest_t.count_ops().get('cx', 0)} CX gates, "
      f"about {honest_t.count_ops().get('cx', 0) / (6 * toffoli_count(n)):.1f}x the "
      f"CX cost of {toffoli_count(n):.0f} Toffolis at 6 CX each.")

The formulas are asymptotic fits and were never meant to be evaluated at $n = 4$, so read that last comparison as an order of magnitude and nothing finer. It does make the right point: the lookup-table construction is already behind at the smallest size that exists, and its cost doubles with every added bit while the fitted curve grows as $n^3$.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12.5, 5))

# left: how the two costs scale
ns = np.array([2 ** k for k in range(2, 13)])
axes[0].loglog(ns, [logical_qubits(x) for x in ns], 'o-', color='#1a5285',
               linewidth=2.5, markersize=7, label='Logical qubits')
axes[0].loglog(ns, [toffoli_count(x) for x in ns], 's-', color='#a02580',
               linewidth=2.5, markersize=7, label='Toffoli gates')
axes[0].axvline(2048, color='k', linestyle='--', linewidth=2, alpha=0.6)
axes[0].text(2048, 3e2, ' RSA-2048', rotation=90, va='bottom', fontsize=10)
axes[0].axvline(n, color='#2d7a4f', linestyle=':', linewidth=2)
axes[0].text(n, 3e2, ' N = 15', rotation=90, va='bottom', fontsize=10, color='#2d7a4f')
axes[0].set_xlabel('modulus size $n$ (bits)')
axes[0].set_ylabel('count')
axes[0].set_title('Gidney-Ekera scaling')
axes[0].grid(alpha=0.3, which='both')
axes[0].legend()

# right: the physical estimate, and what changed between 2019 and 2025
labels = ['Gidney-Ekera\n2019', 'Gidney\n2025']
qubits = [20e6, 1e6]
bars = axes[1].bar(labels, qubits, color=['#1a5285', '#a02580'], width=0.55)
axes[1].set_yscale('log')
axes[1].set_ylabel('physical qubits for RSA-2048')
axes[1].set_title('Same hardware assumptions, six years apart')
axes[1].grid(alpha=0.3, axis='y', which='both')
for bar, val, rt in zip(bars, qubits, ['8 hours', '< 1 week']):
    axes[1].text(bar.get_x() + bar.get_width() / 2, val * 1.15,
                 f'{val:,.0f}\n{rt}', ha='center', fontsize=10)
axes[1].set_ylim(1e5, 1e8)

plt.tight_layout()
plt.show()

print(f"largest device on qBraid today: order 10^2 physical qubits")
print(f"shortfall against the 2025 estimate: {1e6 / 100:,.0f}x")

## Where this leaves the field

Three separate numbers get confused in coverage of quantum factoring, and keeping them apart is most of what this notebook is for.

**What has been factored honestly.** The largest number factored by an implementation that did not use prior knowledge of the answer is 21, and even that implementation took simplifications. Everything larger in the literature is either compiled, or a different algorithm entirely: the 56153 result often cited alongside these used an adiabatic-style minimisation, not Shor's algorithm, and its cost is not governed by Shor's scaling. The honest record has not moved much in over a decade, and the reason is visible in the hardware section above.

**What the hardware needs to be.** Under a million physical qubits at $10^{-3}$ gate error, running coherently for days behind a surface code. Current devices are three to four orders of magnitude short in qubit count and are not error-corrected in the required sense. No published roadmap puts this before the mid-2030s.

**Why the deadline is earlier than the machine.** Encrypted traffic captured now can be stored and decrypted when a machine exists. Mosca's inequality states the problem: if $x$ is how long your data must stay secret, $y$ is how long migration takes, and $z$ is time until a cryptographically relevant machine, you are already late when $x + y > z$. For data with a twenty-year secrecy requirement and a decade-long migration, $x + y$ exceeds most estimates of $z$ regardless of which estimate you pick.

The response is already standardised, not pending. NIST published FIPS 203 (ML-KEM), FIPS 204 (ML-DSA) and FIPS 205 (SLH-DSA) on 13 August 2024, selected HQC as a backup key-encapsulation mechanism in March 2025, and has FN-DSA in draft as FIPS 206. NIST IR 8547 sets the transition schedule: RSA-2048 and ECC P-256 deprecated by 2030, disallowed after 2035.

Which is the honest summary of the field. The algorithm works, the machine does not exist, the estimates for that machine improved twentyfold in six years without any hardware progress, and the migration is underway on a timetable set by that uncertainty rather than by any demonstration. A notebook that factors 15 changes none of it, and the point of building this one carefully is to be able to say precisely why.

## Where to go next

- **Make the period not divide the register.** Factor $N = 21$ with $a = 2$, where $r = 6$ does not divide $2^t$. The peaks broaden, continued fractions stop being decorative, and the success probability per run drops. This is the regime every real instance is in.
- **Replace the lookup table with an adder.** Implement Beauregard's modular multiplier on $2n + 3$ qubits and compare its gate count against `c_mult` at $n = 4, 5, 6$. Find the crossover size, and check it against the $n^3$ fit used above.
- **Sweep the base.** Run the honest circuit for all six $a$ coprime to 15 and record the period, whether the reduction succeeds, and the circuit cost. Two of the six give $r = 2$ and a much cheaper circuit; decide whether reporting only those would be dishonest.
- **Score a claim.** Take a published quantum factoring result and extract three numbers: the period found, the number of non-trivial controlled multiplications, and whether the base was chosen before the factorisation was known. Then say what the result demonstrates.
- **Estimate your own threat model.** Put your organisation's numbers into Mosca's inequality using the 2025 resource estimate and a hardware roadmap you find credible, and work out the year by which migration has to start.

---

**Feedback for the QUEST pedagogy study**

Your feedback informs the IRB-approved QUEST research project on quantum computing pedagogy. Please spend 2 minutes on the following:

1. What was the most useful part of this notebook for your learning?
2. What was the most confusing or under-explained?
3. Which cryptography topic would you most want to see analyzed on real hardware next?

Please submit your responses via the QUEST portal or reply to your instructor.